# Automatic Speech Recognition (ASR) Model Comparison
This notebook demonstrates how to compare several open-source ASR models:
- Wav2Vec2
- HuBERT
- Whisper
- Distil-Whisper

## Step 1: Setup

In [1]:
# Imports for this section

import torch
import torchaudio
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC # (Connectionist Temporal Classification)
from datasets import load_dataset
from jiwer import wer
from pprint import pprint
import numpy as np

/Users/hsamadov/Projects/_umr/nls-voice-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# [1] choose model name (pretrained checkpoint)
model_name = "facebook/wav2vec2-base-960h"

# [2] load processor (feature extractor + tokenizer) and model
processor = Wav2Vec2Processor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)

# [3] load audio sample from HF dataset
dataset = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
sample = dataset[0]
pprint(sample)

audio = sample["audio"]
speech_array = audio["array"]           # numpy array, shape either (samples,) or (channels, samples)
sampling_rate = audio["sampling_rate"]
audio_path = audio.get("path", None)

print(f"Audio sample path: {audio_path}")
# compute number of samples robustly
num_samples = speech_array.shape[-1] if speech_array.ndim == 2 else speech_array.shape[0]
print(f"Duration: {num_samples / sampling_rate:.2f} seconds")

# Reference text
reference_text = sample["text"].lower()
print("Reference text:\n", reference_text)

# handle channels: if 2-D average channels (or take first), if 1-D keep as-is
if isinstance(speech_array, np.ndarray):
    if speech_array.ndim == 1:
        input_audio = speech_array
    elif speech_array.ndim == 2:
        # average channels to mono (shape -> samples)
        input_audio = speech_array.mean(axis=0)
    else:
        raise ValueError(f"Unexpected audio array ndim={speech_array.ndim}")
else:
    # fallback: convert to numpy
    input_audio = np.asarray(speech_array)
    if input_audio.ndim == 2:
        input_audio = input_audio.mean(axis=0)

# ensure float32
input_audio = input_audio.astype(np.float32)

# [4] resample if audio sampling rate != model expected rate (16000 Hz)
target_sr = 16000
if sampling_rate != target_sr:
    tensor_audio = torch.from_numpy(input_audio)
    # torchaudio resampler expects (samples,) or (channels, samples)
    if tensor_audio.ndim == 1:
        resampled = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=target_sr)(tensor_audio)
    else:
        resampled = torchaudio.transforms.Resample(orig_freq=sampling_rate, new_freq=target_sr)(tensor_audio.unsqueeze(0)).squeeze(0)
    input_audio = resampled.numpy()
    sampling_rate = target_sr

# [5] processor expects 1-D array or list of arrays
inputs = processor(input_audio, sampling_rate=sampling_rate, return_tensors="pt", padding=True)

# [6] move model and inputs to GPU if available (optional)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
for k, v in inputs.items():
    inputs[k] = v.to(device)

# [7] forward pass (inference) — get logits 
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # shape: (batch, time, vocab_size)

# [8] greedy decode: take argmax across vocab dim -> predicted token IDs
predicted_ids = torch.argmax(logits, dim=-1)

# [9] convert token IDs -> text
transcription = processor.batch_decode(predicted_ids)[0].lower()
print("Transcription:", transcription)

# [10] Evaluate with WER
error = wer(reference_text, transcription)
print(f"Word Error Rate (WER): {error:.3f}")


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'audio': {'array': array([0.00238037, 0.0020752 , 0.00198364, ..., 0.00042725, 0.00057983,
       0.0010376 ]),
           'path': '1272-128104-0000.flac',
           'sampling_rate': 16000},
 'chapter_id': 128104,
 'file': '/Users/sanchitgandhi/.cache/huggingface/datasets/downloads/extracted/aad76e6f21870761d7a8b9b34436f6f8db846546c68cb2d9388598d7a164fa4b/dev_clean/1272/128104/1272-128104-0000.flac',
 'id': '1272-128104-0000',
 'speaker_id': 1272,
 'text': 'MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD '
         'TO WELCOME HIS GOSPEL'}
Audio sample path: 1272-128104-0000.flac
Duration: 5.86 seconds
Reference text:
 mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
Transcription: mister quilter is the apostle of the middle classes and we are glad to welcome his gospel
Word Error Rate (WER): 0.000


# Comparing 3 different speakers

In [3]:
from transformers import pipeline
from jiwer import wer
import time
from pprint import pprint
from datasets import Dataset, load_dataset, Audio

In [4]:
audio_dataset = Dataset.from_dict({"audio": ["group_5/01__common_voice_en_18296536.mp3",
                                             "group_5/02__common_voice_en_18297346.mp3",
                                             "group_5/03__common_voice_en_18298153.mp3"],
                                "text": ["AN UNUSUAL SENSATION ON THE OPPOSITE SIDE OF MY STOMACH",
                                         "AN UNUSUAL SENSATION ON THE OPPOSITE SIDE OF MY STOMACH",
                                         "AN UNUSUAL SENSATION ON THE OPPOSITE SIDE OF MY STOMACH"]}
                                             ).cast_column("audio", Audio())
# pprint(audio_dataset[0])
audio_path = audio_dataset[1]["audio"]["path"]

In [5]:
audio_dataset[1]

{'audio': {'path': 'group_5/02__common_voice_en_18297346.mp3',
  'array': array([0.00000000e+00, 1.00288957e-17, 1.42367528e-17, ...,
         1.11695908e-06, 1.47833703e-06, 7.40709368e-07]),
  'sampling_rate': 48000},
 'text': 'AN UNUSUAL SENSATION ON THE OPPOSITE SIDE OF MY STOMACH'}

# Define helper for running ASR

In [6]:
def run_asr(model_name, asr_pipeline):
    """Run ASR pipeline and measure time."""
    start = time.time()
    result = asr_pipeline(audio_path)
    elapsed = time.time() - start
    text = result["text"].lower()
    return text, elapsed

# Wav2Vec 2.0


In [7]:

wav2vec_asr = pipeline("automatic-speech-recognition", model="facebook/wav2vec2-base-960h")
wav2vec_text, wav2vec_time = run_asr("wav2vec2-base-960h", wav2vec_asr)


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use mps:0


# HuBERT 

In [8]:

hubert_asr = pipeline("automatic-speech-recognition", model="facebook/hubert-large-ls960-ft")
hubert_text, hubert_time = run_asr("hubert-large-ls960-ft", hubert_asr)


Device set to use mps:0



# Whisper


In [17]:

whisper_asr = pipeline("automatic-speech-recognition", model="openai/whisper-small")
whisper_text, whisper_time = run_asr("whisper-small", whisper_asr)


Device set to use mps:0



# Distil-Whisper


In [10]:

dwhisper_asr = pipeline("automatic-speech-recognition", model="distil-whisper/distil-small.en")
dwhisper_text, dwhisper_time = run_asr("distil-whisper", dwhisper_asr)


Device set to use mps:0


# NVIDIA NeMo


In [ ]:
# !pip install nemo_toolkit[all]

In [ ]:
# import nemo.collections.asr as nemo_asr

In [ ]:

# nemo_model = nemo_asr.models.ASRModel.from_pretrained("stt_en_conformer_transducer_small")
# start = time.time()
# nemo_text = nemo_model.transcribe([audio_path])[0][0].lower()
# nemo_time = time.time() - start



# Compare results


In [11]:

results = {
    "Wav2Vec2": (wav2vec_text, wav2vec_time),
    "HuBERT": (hubert_text, hubert_time),
    "Whisper": (whisper_text, whisper_time),
    "Distil-Whisper": (dwhisper_text, dwhisper_time),
    # "NeMo": (nemo_text, nemo_time),
}

print("\n--- TRANSCRIPTION RESULTS ---")
for name, (text, t) in results.items():
    print(f"\n{name} (time: {t:.2f}s):")
    print(text)


--- TRANSCRIPTION RESULTS ---

Wav2Vec2 (time: 0.49s):
head an doum sur says he should undou buse sile my stumwat

HuBERT (time: 0.85s):
an unusual sensation on the opposite side of my stomac

Whisper (time: 9.28s):
 keadaan yang tidak neutral di sisi yang sama di atas kawasan saya.

Distil-Whisper (time: 6.49s):
 an unneutral sensation on the opposite side of my stomach.



# Compute WER

In [12]:
reference_text = "AN UNUSUAL SENSATION ON THE OPPOSITE SIDE OF MY STOMACH".lower()

print("\n--- WORD ERROR RATES ---")
for name, (text, _) in results.items():
    error = wer(reference_text, text)
    print(f"{name:15s} WER: {error:.3f}")



--- WORD ERROR RATES ---
Wav2Vec2        WER: 1.000
HuBERT          WER: 0.100
Whisper         WER: 1.200
Distil-Whisper  WER: 0.200


# Discussion 

1. Which model had the lowest WER?
2. Which was fastest?
3. Do larger models always perform better?
4. How would noise or accents affect performance?
5. Which file type is ideal for ASR (mp3, wav, flac...)?
6. Try running models on your own recordings!

In [13]:
import pandas as pd
from jiwer import wer  

models = {
    "Wav2Vec2": wav2vec_asr,
    "HuBERT": hubert_asr,
    "Whisper": whisper_asr,
    "Distil-Whisper": dwhisper_asr,
    # "NeMo": nemo_model
}


In [14]:
rows = []
for item in audio_dataset:
    audio_path = item["audio"]["path"]
    reference = item.get("text")  # may be None for unlabeled audio
    for name, pipeline_obj in models.items():
        start = time.time()
        out = pipeline_obj(audio_path)
        elapsed = time.time() - start
        text = out.get("text", "") .lower()
        err = wer(reference.lower(), text) if (reference and isinstance(reference, str)) else None
        rows.append({
            "audio_path": audio_path,
            "model": name,
            "transcript": text,
            "time_s": elapsed,
            "wer": err
        })

In [15]:
# Create and save DF 
df = pd.DataFrame(rows)
df.to_csv("transcripts_by_model.csv", index=False) 

In [16]:
df

,audio_path,model,transcript,time_s,wer
0,group_5/01__common_voice_en_18296536.mp3,Wav2Vec2,an unusual sensation on the opposite side of m...,0.712578,0.0
1,group_5/01__common_voice_en_18296536.mp3,HuBERT,an unusual sensation on the opposite side of m...,0.333024,0.0
2,group_5/01__common_voice_en_18296536.mp3,Whisper,an unusual sensation on the opposite side of ...,1.116226,0.1
3,group_5/01__common_voice_en_18296536.mp3,Distil-Whisper,an unusual sensation on the opposite side of ...,0.375589,0.1
4,group_5/02__common_voice_en_18297346.mp3,Wav2Vec2,head an doum sur says he should undou buse sil...,0.100484,1.0
5,group_5/02__common_voice_en_18297346.mp3,HuBERT,an unusual sensation on the opposite side of m...,0.137282,0.1
6,group_5/02__common_voice_en_18297346.mp3,Whisper,keadaan yang tidak neutral di sisi yang sama ...,0.932576,1.2
7,group_5/02__common_voice_en_18297346.mp3,Distil-Whisper,an unneutral sensation on the opposite side o...,0.360999,0.2
8,group_5/03__common_voice_en_18298153.mp3,Wav2Vec2,an unusual sensation on the opposite side of m...,0.333635,0.0
9,group_5/03__common_voice_en_18298153.mp3,HuBERT,an unusual sensation on the opposite side of m...,0.335599,0.0
